# TrajAI State Tests

Experimentation with my libraries to ensure that functions work as intended.


This code is not professional-grade, but is provided to the curious user as a loose reference with no warranties.

In [1]:
import torch
import traj_ai as ta

In [2]:
# Define some simple states for testing, in the form of the training data
traj_reduced = torch.Tensor([
    # Time frame 1
    [
        # Particle 1
        [
            1, # x
            0, # y
            0, # vx
            1, # vy
            37, # attr1
            42 # attr2
        ],
        # Particle 2
        [
            -10, # x
            2, # y
            1, # vx
            1, # vy
            99, # attr1
            5 # attr2
        ]
    ],
    # Time frame 2
    [
        # Particle 1
        [
            9, # x
            1, # y
            -2, # vx
            5, # vy
            38, # attr1
            41 # attr2
        ],
        # Particle 2
        [
            -9, # x
            3, # y
            1, # vx
            0, # vy
            100, # attr1
            4 # attr2
        ]
    ]
])
num_past = 1

In [3]:
# Transposers
state_composer = ta.StateComposer(num_past)
state_decomposer = ta.StateDecomposer(num_past)

In [4]:
traj = state_composer(traj_reduced)
traj

tensor([[[  9.,   1.,   1.,   0.,  -2.,   5.,   0.,   1.,  38.,  41.],
         [ -9.,   3., -10.,   2.,   1.,   0.,   1.,   1., 100.,   4.]]])

In [5]:
state_decomposer(traj.squeeze(0))

tensor([[  9.,   1.,  -2.,   5.,  38.,  41.],
        [ -9.,   3.,   1.,   0., 100.,   4.]])

In [6]:
# Relaters
relater = ta.NormalTangentialDistanceObjectiveStateRelater(num_past)

In [7]:
influencers = traj[0, :, :]
influencees = torch.roll(influencers, shifts=1, dims=0)
influencees

tensor([[ -9.,   3., -10.,   2.,   1.,   0.,   1.,   1., 100.,   4.],
        [  9.,   1.,   1.,   0.,  -2.,   5.,   0.,   1.,  38.,  41.]])

In [8]:
relater(influencers, influencees)

tensor([[-1.8000e+01,  2.0000e+00, -6.3640e+00,  9.1924e+00,  3.0000e+00,
         -5.0000e+00,  7.0711e-01, -7.0711e-01,  1.8111e+01,  1.1180e+01],
        [-8.5420e+00, -1.5970e+01, -2.0000e+00, -1.1000e+01,  5.7566e+00,
          9.2848e-01,  4.3711e-08,  1.0000e+00,  1.8111e+01,  1.1180e+01]])

In [9]:
# Transcoders
transcoder = ta.NormalTangentialObjectiveStateTranscoder(num_past)

In [10]:
transcoder(traj.squeeze(0))

tensor([[ -2.0426,  -7.7992,   5.4772,   0.9285,  -0.3714,  38.0000,  41.0000],
        [  1.0000,   1.0000,   5.4772,   1.0000,   1.0000, 100.0000,   4.0000]])

In [12]:
# Updaters
velocity_updater = ta.LocalVelocityUpdater(dt=1, num_past=num_past)
acceleration_updater = ta.LocalAccelerationUpdater(dt=1, num_past=num_past)

In [14]:
velocity_updater(
    traj[0, 0:1, :],
    torch.tensor([[1.0, 0.0]])
)

tensor([[ 8.6286,  1.9285,  9.0000,  1.0000, -0.3714,  0.9285, -2.0000,  5.0000,
         38.0000, 41.0000]])